# Q7: Intel Image Classification using Transfer Learning (PyTorch)

## Objective
Build an image classification model to classify outdoor scene images into the following classes:

1. Buildings
2. Forest
3. Glacier
4. Mountain
5. Sea
6. Street

Dataset: https://www.kaggle.com/datasets/puneet6060/intel-image-classification

# Part A: Data Preparation (25 Marks)

## Step 1: Install Kaggle and Download Dataset

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip intel-image-classification.zip

## Dataset Structure
```
seg_train/
    buildings
    forest
    glacier
    mountain
    sea
    street

seg_test/
    buildings
    forest
    glacier
    mountain
    sea
    street
```

## Step 2: Import Libraries

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

from torchvision import transforms, datasets
from torch.utils.data import DataLoader

## Step 3: Data Augmentation and Preprocessing

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

## Step 4: Load Dataset

In [ ]:
train_dataset = datasets.ImageFolder(
    "seg_train/seg_train",
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    "seg_test/seg_test",
    transform=test_transform
)

In [ ]:
class_names = train_dataset.classes
print(class_names)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# Part B: Transfer Learning (10 Marks)

In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = models.resnet18(pretrained=True)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    6
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [ ]:
train_loss = []
train_acc = []

val_loss = []
val_acc = []

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss/len(train_loader)
    epoch_acc = 100*correct/total

    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss={epoch_loss:.4f}, "
        f"Accuracy={epoch_acc:.2f}%"
    )

# Part C: Model Evaluation and Inference (10 Marks)

In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

test_accuracy = 100*correct/total

print("Test Accuracy:", test_accuracy)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_acc)
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_loss)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names
    )
)

# Detecting Overfitting

Overfitting can be identified when:

- Training accuracy keeps increasing.
- Validation accuracy stops improving.
- Validation loss starts increasing.
- Large gap exists between training and validation accuracy.

Example:

Train Accuracy = 98%  
Validation Accuracy = 82%

Solutions:

- Data augmentation
- Dropout
- Early stopping
- Reduce model complexity
- More data

In [ ]:
def save_model(model, path):
    torch.save(model.state_dict(), path)

save_model(
    model,
    "Q7_trained_model.pth"
)

In [ ]:
def load_model(path):

    model = models.resnet18(pretrained=False)

    num_features = model.fc.in_features

    model.fc = nn.Linear(
        num_features,
        6
    )

    model.load_state_dict(
        torch.load(path)
    )

    model.eval()

    return model

In [ ]:
from PIL import Image

def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    image = test_transform(image)
    image = image.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image)

        _, pred = torch.max(outputs,1)

    return class_names[pred.item()]

In [ ]:
predict_image("sample.jpg")

# Files to Submit

- Q7_Intel_Image_Classification.ipynb
- Q7_trained_model.pth
- Q7_output_screenshots.pdf
- Q7_confusion_matrix.png
- Q7_accuracy_loss_curves.png

# Workflow Diagram

Dataset → Preprocessing → Data Augmentation → Transfer Learning (ResNet18) → Training (10 Epochs) → Evaluation → Confusion Matrix → Save Model → Inference on New Images